# Simulators and Real Backends in Modern Qiskit

A quick reference notebook showing how the *backend story* has changed since the original course was recorded:

| Old API (Qiskit ≤ 0.45) | New API (Qiskit 1.0 / 2.x) |
|---|---|
| `from qiskit import IBMQ` | `from qiskit_ibm_runtime import QiskitRuntimeService` |
| `IBMQ.save_account(token)` | `QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=...)` |
| `IBMQ.load_account()` | `service = QiskitRuntimeService()` |
| `Aer.backends()` | `AerSimulator().available_methods()` |
| `Aer.get_backend("qasm_simulator")` | `AerSimulator()` |
| `IBMQ.get_provider("ibm-q")` | The `service` itself is the provider |
| `provider.backends()` | `service.backends()` |
| `backend.name()`  / `backend.properties().qubits` | `backend.name`  / `backend.num_qubits` |

In [1]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

## Saving your IBM Quantum API token

The `IBMQ.save_account(...)` workflow has been **removed**. You now save your account through the `QiskitRuntimeService` class. Run this **once** – the credentials get written to `~/.qiskit/qiskit-ibm.json`.

> Make sure `ibmapi.txt` contains *only* your API key (and nothing else – no quotes, no whitespace).

In [2]:
# OLD (removed in Qiskit 1.0):
#   from qiskit import IBMQ
#   IBMQ.save_account(open("ibmapi.txt", "r").read())
#
# NEW – save your IBM Quantum account once with QiskitRuntimeService.
# After this is run a single time, you never need to call save_account again
# from the same machine.
#QiskitRuntimeService.save_account(
#    channel="ibm_quantum_platform",
#    token=open("ibmapi.txt", "r").read().strip(),
#    overwrite=True,
#)

### Loading the account on subsequent runs

Once your account is saved, every later session just needs `QiskitRuntimeService()`.

In [3]:
# Load the saved account.
service = QiskitRuntimeService()
print("Account loaded:", service.active_account())

qiskit_runtime_service.__init__:WARNING:2026-05-02 14:34:47,151: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


Account loaded: {'channel': 'ibm_quantum_platform', 'url': 'https://cloud.ibm.com', 'token': 'tl15Sa1IfCMvS4I5N0hAjtqUTBmQ24kdsuj6760O_UQ2', 'verify': True, 'private_endpoint': False}


## Local simulators – Aer

Aer used to expose a *list* of named simulators (`qasm_simulator`, `statevector_simulator`, …). Now there is one class, `AerSimulator`, that switches between simulation **methods** internally.

In [4]:
# In modern Qiskit there is no `Aer.backends()` listing. The single
# `AerSimulator` class can switch between several simulation methods:
print("AerSimulator methods:", AerSimulator().available_methods())

AerSimulator methods: ('automatic', 'statevector', 'density_matrix', 'stabilizer', 'matrix_product_state', 'extended_stabilizer', 'unitary', 'superop')


## Real-hardware providers

The legacy `IBMQ.get_provider("ibm-q")` has been retired – your `QiskitRuntimeService` instance now plays both roles.

In [5]:
# In modern Qiskit Runtime there's no separate provider –
# `QiskitRuntimeService` itself is the gateway to all your backends.
# We keep the variable name `provider` so the rest of the notebook
# keeps reading naturally.
provider = service

### Listing the real backends

`provider.backends()` returns every QPU on your account.

In [6]:
provider.backends()

qiskit_runtime_service.backends:WARNING:2026-05-02 14:34:53,027: Loading instance: open-instance, plan: open


[<IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_kingston')>]

### Pretty-printing each backend

Note the two small API tweaks compared with the old notebook:

- `backend.name()` (a method) → `backend.name` (a property).
- `len(backend.properties().qubits)` → `backend.num_qubits`.

In [7]:
for backend in provider.backends():
    try:
        qubit_count = backend.num_qubits  # was: len(backend.properties().qubits)
    except Exception:
        qubit_count = "?"
    # In modern Qiskit `backend.name` is a property, not a method.
    print(f"{backend.name} : {backend.status().pending_jobs} pending jobs & {qubit_count} qubits")

qiskit_runtime_service.backends:WARNING:2026-05-02 14:34:56,037: Loading instance: open-instance, plan: open


ibm_fez : 14 pending jobs & 156 qubits
ibm_marrakesh : 0 pending jobs & 156 qubits
ibm_kingston : 0 pending jobs & 156 qubits
